### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 8

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 8
Seed set to 42


random seed set to 8
numpy seed set to 8
torch seed set to 8
lightning seed set to 8
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


In [4]:
interaction_info_df["actorID"]

0         [jack_lemmon, walter_matthau, annmargret, burg...
1                  [[RARE], [RARE], [RARE], [RARE], [RARE]]
2         [mel_gibson, sophie_marceau, patrick_mcgoohan,...
3         [[RARE], laura_linney, ernie_hudson_jr, tim_cu...
4         [antonio_banderas, salma_hayek, 1142520-joaqui...
                                ...                        
478399             [[RARE], [RARE], [RARE], [RARE], [RARE]]
478400    [greg_kinnear, toni_collette, steve_carell, pa...
478401    [leonardo_di_caprio, matt_damon, jack_nicholso...
478402    [penelope_cruz, ben_kingsley, dennis_hopper, p...
478403    [john_hurt, richard_burton, [RARE], cyril_cusa...
Name: actorID, Length: 478404, dtype: object

### Prepare Train/Valid/Test Set

In [5]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.64,
    val_ratio=0.16,
    test_ratio=0.20,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.64 : 0.16 : 0.2):
train: 305206 (63.8%)
valid: 75578 (15.8%)
test: 97620 (20.41%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.547669
1    0.452331
Name: proportion, dtype: float64
valid label
0    0.606737
1    0.393263
Name: proportion, dtype: float64
test label
0    0.590914
1    0.409086
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [6]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [7]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [8]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 305206
  Num of positive interactions: 138054 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 276107], edge_label=[138054])
Edge Index: tensor([[    0,     0,     0,  ..., 10448, 10452, 10453],
        [ 2089,  2200,  2315,  ...,  1760,  1645,  1760]])


#### Prepare train/valid triplet data

In [9]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5) # TODO: use TripletDataset for validation as well
train_triplet_df.head(1)

Original data count (positive samples): 138054
Num of triplets: 138054(pos samples) * 5(negative sampled items) = 690270
Original data count (positive samples): 29722
Num of triplets: 29722(pos samples) * 5(negative sampled items) = 148610


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,neg_genre_idx
0,0,1082,2876,"[2267, 1401, 582, 998, 1847]",37,360,"[2, 3, 17, 18, 0, 0, 0, 0]","[147, 1, 1, 1, 1]",37,1,"[6, 0, 0, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [10]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_df)
valid_dataset = TripletDataset(valid_triplet_df)  # TODO: use TripletDataset for validation as well
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)


train data count: 690270
valid data count: 148610


### Configure Model (LightningModule)

In [11]:
from lightning_models.ngcf_v2 import NGCFRecV2

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-3

model = NGCFRecV2(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
)


Seed set to 42


### Configure Trainer and Experiment

In [12]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "ngcf-exp"
VERSION = "stat_test"
RUN_NAME = f"{RANDOM_SEED}"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_bpr_loss",
    monitor_mode="min",
    min_delta=0.01,
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-reg_weight={REG_WEIGHT}",
)

In [13]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='gpu',  # or 'auto', 'gpu'
    devices=[1], # if gpu is available
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [14]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/ngcf-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name       | Type    | Params | Mode 
-----------------------------------------------
0 | ngcf_model | NGCF    | 694 K  | train
1 | bpr_loss   | BPRLoss | 0      | train
2 | reg_loss   | EmbLoss | 0      | train
-----------------------------------------------
694 K     Trainable params
0         Non-trainable params
694 K     Total params
2.778     Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 8, global step 6075: 'val_bpr_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_bpr_loss did not improve in the last 5 records. Best score: 0.549. Signaling Trainer to stop.
Epoch 9, global step 6750: 'val_bpr_loss' was not in top 1


🏃 View run 8 at: http://140.112.106.216:3683/#/experiments/6/runs/8915579417874b6a87237e6db9bdcc61
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/6


### Inference

#### Prepare prediction pool for inference/testing

In [ ]:
# NOTE: Prepare prediction pool to evaluate the model
seen_df = pd.concat([encoded_train_df, encoded_valid_df], ignore_index=True)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, seen_df, K=-1)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 8397, negative sampled to -1 items for each user
Num of interactions: 2064(users) * -1(items) = 16952432


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
16952427,2063,8392,0,"[2214, 1834, 1886, 2036, 1214]",37,1,"[6, 0, 0, 0, 0, 0, 0, 0]"
16952428,2063,8393,0,"[1070, 110, 1, 1, 1]",12,308,"[9, 0, 0, 0, 0, 0, 0, 0]"
16952429,2063,8394,0,"[1782, 1, 1, 1, 0]",36,365,"[4, 6, 0, 0, 0, 0, 0, 0]"
16952430,2063,8395,0,"[1, 1733, 935, 24, 1]",37,443,"[9, 17, 18, 0, 0, 0, 0, 0]"
16952431,2063,8396,0,"[1, 1, 0, 0, 0]",17,1,"[9, 0, 0, 0, 0, 0, 0, 0]"


In [ ]:
from torch.utils.data import DataLoader
from common.datasets import UserItemPairDataset

test_dataset = UserItemPairDataset(prediction_pool_df)
print("test data count:", len(test_dataset))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


test data count: 16952432


In [ ]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "ngcf-exp"
# best_model_checkpoint_path = ""
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = NGCFRecV2.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.15267327427864075    │
│        test_ndcg20        │    0.17651647329330444    │
│        test_ndcg5         │    0.12129464000463486    │
│     test_precision10      │   0.043943800032138824    │
│     test_precision20      │   0.036143410950899124    │
│      test_precision5      │    0.04806201532483101    │
│       test_recall10       │    0.02749955840408802    │
│       test_recall20       │   0.043522801250219345    │
│       test_recall5        │   0.015521940775215626    │
└───────────────────────────┴───────────────────────────┘

🏃 View run 7 at: http://140.112.106.216:3683/#/experiments/6/runs/932873c6d2ea40edb08efe7a1e7747ba
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/6


[{'test_ndcg5': 0.12129464000463486,
  'test_ndcg10': 0.15267327427864075,
  'test_ndcg20': 0.17651647329330444,
  'test_precision5': 0.04806201532483101,
  'test_precision10': 0.043943800032138824,
  'test_precision20': 0.036143410950899124,
  'test_recall5': 0.015521940775215626,
  'test_recall10': 0.02749955840408802,
  'test_recall20': 0.043522801250219345}]

In [ ]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.121295,0.015522,0.048062,0.152673,0.027500,0.043944,0.176516,0.043523,0.036143
std,595.969798,0.269242,0.053719,0.111987,0.264965,0.070411,0.083444,0.253952,0.088948,0.059133
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1547.250000,0.000000,0.000000,0.000000,0.315465,0.027778,0.100000,0.333333,0.058824,0.050000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.700000,1.000000,1.000000,0.450000


In [ ]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True, rescale=True)

# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:07<00:00, 292.28it/s]


candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:04<00:00, 493.97it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.087311,0.960728,0.084317,0.800156,0.483128
std,595.969798,0.046311,0.049719,0.075055,0.103624,0.043045
min,0.000000,0.000000,0.279260,0.000000,0.280177,0.266052
25%,515.750000,0.052547,0.953607,0.012820,0.745923,0.457338
50%,1031.500000,0.086442,0.975709,0.075255,0.820695,0.486656
75%,1547.250000,0.118412,0.986086,0.128126,0.878455,0.512312
max,2063.000000,0.276040,1.000000,0.415294,0.976847,0.615182


In [ ]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(eval_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.216675
std,20797.975208,0.055475
min,75.000000,0.061627
25%,17798.500000,0.176935
50%,35054.000000,0.213267
75%,53331.000000,0.254229
max,71534.000000,0.439815


In [ ]:
import joblib
dir = "artifacts/ngcf"
prefix = "stat_test"
if not os.path.exists(dir):
    os.makedirs(dir)

joblib.dump(eval_df, os.path.join(dir, f"{prefix}_eval_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

['artifacts/ngcf/stat_test_feature_engineer.pkl']

In [ ]:
embedding_path = "embeddings/ngcf/"
prefix = "stat_test"
if not os.path.exists(embedding_path):
    os.makedirs(embedding_path)

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")